# This notebook will serve as a way to clean the remaining txt that will be used for char generation

In [130]:
import re

In [131]:
with open("corpus_RNN_Médine.txt", "r", encoding="utf-8", errors="replace") as f:
    corpus = f.read()

First we will remove the indication of featuring of the corpus

In [132]:
corpus = re.sub(r">(True|False)\n", ">\n", corpus)

As the task will be char generation, first I will let indications such as INTRO, etc... in order to maybe have a RNN that can maybe identify what type of text came after Refrain (more repetitive) or after Couplet

In [133]:
corpus[:200]

"<BEGINNING>\n<INTRO>\nC'est nous l'Grand Paris  C'est nous l'Grand Paris \n<END_SECTION>\n<COUPLET>\nLa banlieue, c'est des pâquerettes sur un tas d'fumier \nLes voix du ghetto sont auto-tunées \n<END_SECTIO"

In [134]:
print(len(set(corpus)),sorted(set(corpus)))

130 ['\n', ' ', '!', '#', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\x80', '\x99', '«', '·', '»', 'À', 'Ç', 'È', 'É', 'Ê', 'Ô', 'Ö', 'à', 'á', 'â', 'ã', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'í', 'î', 'ï', 'ñ', 'ô', 'ö', 'ù', 'û', 'ü', 'Ā', 'ā', 'ğ', 'ō', 'Œ', 'œ', 'ū', 'ʿ', 'е', '–', '‘', '’', '“', '”', '…', '\ufeff']


## Corpus cleaning

Some parts are empty of lyrics because they were used just for training on the structure

In [135]:
print(re.findall(r"<.*>\n<END_SECTION>", corpus)[:5])

corpus = re.sub(r"<.*>\n<END_SECTION>\n","",corpus)


['<PONT>\n<END_SECTION>', '<REFRAIN>\n<END_SECTION>', '<COUPLET>\n<END_SECTION>', '<PONT>\n<END_SECTION>', '<COUPLET>\n<END_SECTION>']


End_Section can be replaced by a specific character that isn't in the corpus such as §

In [136]:
corpus = re.sub(r"\n<END_SECTION>\n","\n§\n", corpus)

In [137]:
re.findall("\s+.*\n§\n", corpus)[:5]

["\nC'est nous l'Grand Paris  C'est nous l'Grand Paris \n§\n",
 ' \nLes voix du ghetto sont auto-tunées \n§\n',
 "\n \nPapy t'a défendu, est-c'que tu t'en souviens ? \n§\n",
 "\npour qu'la rafleuse te touche \n§\n",
 " \nC'est nous, on braque Paris \n§\n"]

Indication of beginning and ending of a song isn't required, the model learn only the lyrics structure inside a part not a song structure

In [138]:
corpus = re.sub(r"<BEGINNING>\n","", corpus)
corpus = re.sub(r"<END>\n","", corpus)

Rapid analysis of the corpus shows that some characters or words are alone on a line, most of the time it's because they are between two parts of lyrics that contains annotations on Genius and so they aren't in the same container in html

In [139]:
print(len(re.findall(r"\n,\n", corpus)))
corpus = re.sub(r"\n,\n", ",", corpus)

81


In [140]:
print(re.findall(r"\n\w+\n", corpus)[:10],len(re.findall(r"\n\w+\n", corpus))) 

['\nCheck\n', '\nf\n', '\nAvant\n', '\nTa\n', '\nvitriole\n', '\nphallocrate\n', '\nDes\n', '\nMic\n', '\ninvisible\n', '\ncrouille\n'] 141


In [141]:
corpus = re.sub(r"\n([a-z]\w+)\n", r"\1\n", corpus) #If lowercase, we assume that the sentence wasn't finished
corpus = re.sub(r"\n([A-Z]\w+)\n", r"\n\1", corpus) #If uppercase, we assume we are at the beginning of the sentence

In [142]:
print(re.findall(r"\n\w+\n", corpus)[:10],len(re.findall(r"\n\w+\n", corpus))) 

['\nf\n', '\nprétorien\n', '\nLeFLN\n', '\n1429\n', '\nMédine\n', '\nÉcoutes\n', '\nStopMédine\n', '\nAminAmin\n', '\nW\n', '\nMédineBrav\n'] 16


### Characters : <>

In [143]:
set(re.findall(r"\<.*\>", corpus)) #No need to modify this

{'<COUPLET>', '<INTRO>', '<OUTRO>', '<PONT>', '<REFRAIN>'}

### Characters : ()

In [144]:
print(re.findall(r"\(.*?\)", corpus, flags=re.DOTALL)[:5])
corpus = re.sub(r"\(.*?\)", "", corpus, flags=re.DOTALL)

['(\nbang, bang\n)', '(\nservir de modèle\n)', '(\ncrache comme venom\n)', '(\nla terre des colons\n)', '(\nle sang va couler\n)']


In [145]:
re.findall(r"\(.*?\)", corpus, flags=re.DOTALL)

[]

### Character : *

In [146]:
print(re.findall(r"\*.*", corpus))

corpus = re.sub(r"\*", "", corpus, flags=re.DOTALL)

['*Spartiate, quel est votre métier? Aouh, aouh, aouh!*', '*Détonation*', '*Femme pleurant au téléphone*', "*Femme expliquant la situation avant qu'une explosion retentisse*"]


### Characters : « »

In [147]:
print(len(re.findall("«.*?»", corpus)),re.findall("«.*?»", corpus)[:5])
#This lyrics are just citation such as TV journals having a reportage on Medine
corpus = re.sub("«.*?»", "", corpus)

51 ['« Nietzsche est mort »', "« Vous avez rencontré un artiste du Havre, très connu sur les réseaux sociaux, il a utilisé énormément Twitter, il s'appelle Médine, il est rappeur et il a mis en ligne début janvier un morceau polémique sur la laïcité qui n'a pas très bien été compris... »", "« Il a débuté dans la carrière par une apologie musicale des attentats du 11 Septembre et de Ben Laden. Il s'est donc retrouvé interdit d'antenne »", "«...Page 32 de Marianne, y'a un autre papier: Les Sales Voies du Jihad notamment vous citez le rapport... le rappeur pardon Médine qui se produisait... »", "«... Ce clip... parce qu'en fait, on s'aperçoit que le marketing du djihadisme aujourd'hui est...»"]


### Characters : “ ”

In [148]:
print(re.findall(".*”", corpus))

corpus = re.sub("”", '"', corpus)
corpus = re.sub("“", '"', corpus)

["J'essaie d'leur rendre la guerre un peu moins horrible Comme dans “La vie est belle”", "J'te souhaite la bienvenue au Havre , autrefois appelé “Franciscopolis”"]


### Character : …

In [149]:
print(re.findall(".*….*", corpus))

corpus = re.sub("…", "...", corpus)

['La terre fleurit dès que je l’ai foulée…', 'Réponse: Trois petits points de suspension…', 'Message de Boudj Samedi 28 octobre à 14h35 Ouais Médine c’est Boudj, eh poto, qu’est-ce qui s’passe? Ça fait deux semaines que j’essaye de te joindre, tu réponds pas Donc vas-y, tiens-moi au courant si y a une galère ou quoi Même Salsa il répond pas, donc on commence à s’inquiéter un petit peu On s’demande si vous avez pas été enrôlé dans les combattants d’Al Qaïda Vas-y décroche le téléphone on a du taf poto Et vas-y, euh, rase-toi la barbe…', 'une guerre avant une autre, et un mort après l’autre Un empire un continent et une race contre une autre Récit imaginaire, mythologie du Minotaure Hercule contre centaure; Achille contre Hector Comme les responsables…']


### Characters : ‘ ’

In [150]:
print(re.findall(".*‘.*",corpus))

corpus = corpus.replace("‘", "'")

['C‘est pour les Russel Crowe qui recèlent trop', 'Où l‘on s‘achète un pavillon avec une sonnerie polyphonique', 'On est des tas, décrétés ennemis d’État État des lieux, vendetta,  et Beretta On respecta, comme Franklin Aretha Mais l‘histoire nous rattrapa, car troqués en pesetas', 'Qui ser-‘cra, même le pire des scards-la sant-cri', "C‘est dur de voir l'adolescence en manque de souffle"]


In [151]:
print(len(re.findall(".*’.*",corpus)),re.findall(".*’.*",corpus)[:5])

corpus = re.sub("’", "'", corpus)

829 ["J'viens pour libérer les quartiers, pas d'quartier pour Quartiers Libres T'as la vision d'un journaliste alors je t’appelle le presbyte Une presse qui n'a de street que le nom de domaine de son blog", 'T’as pas de grands frères, de grands hommes auxquels t’identifier T’as pas de grandes guerres, de grandes causes auxquelles te sacrifier Pas de gangsters, de grands guns pour s’authentifier Sous nos grands airs de grandes gueules on vient pour s’unifier', ' ou bien c’est juste un grand quartier', "J’m'en bats les steaks de leurs articles", "J'fais des fautes d’orthographe même à l'oral, j'ai des fous rires nerveux dans nos mémorials"]


In [152]:
re.findall(".*’.*",corpus)

[]

### Characters : е (different than normal e)

In [153]:
print(re.findall(".*е.*", corpus)[:5])

corpus = re.sub("е", "e", corpus)

["J'ai dû m'éloigner d'leurs mosquées pour mе rapprocher de Dieu Ils s'battеnt pour les pouvoirs comme sur l'rrain-te pour un 10 eu' Beaucoup d'foi pour haïr mais pas assez pour aimer Préfèrent élever leurs âmes que d'faire des gosses bien élevés", "Mais faudra vraiment être sûr de toi parce que c'que tu diras sur scène, le public s'еn emparera et s'y connеctera. Les gens s'approprient en tes mots. Ils s'en empareront. Et à partir de ce moment-là, ils s'envoleront pour toujours", "J'vais t'mettre plus haut que toutes les damеs", "Dans tout mon cœur, t'as mis du dramе", 'Ce sera jamais nous contre eux, les mauvaises hеrbes sont des fleurs']


### Character : \#

In [154]:
print(re.findall(".*#.*", corpus))

corpus = re.sub("#", "",  corpus)

['RT, #dièse, NGRTD', 'Frappe les PAO,gagne par KO! #MannyPacquiao', "La France j'y suis, j'y reste. Anti-colonial #JeanJaurès", "On n'innocente pas le petit Ahmed avec un #NotInMyName", "Y'a pas qu'en concert qu'on les fait jumper Comme Kanye j'inclus les handicapés #Oups", 'Leur engagement disparaît comme le tweet pour Gaza de Rihanna #FreePalestine #AuFreestyleDeSky', "Depuis que la zik s'écoute que sur les sites bientôt ton boss s'appelle Fif Tobossi #Oups", "Le seul slogan qu'on voit c'est hashtag #FreeLacrim", "OrelsanJe porte un toast à la mort de l'industrieSortez les 8'6, on vient fêter la fin du disqueÉcouter la radio c'est devenu un suppliceSauf que j'aime pas non plus les putains de puristesMusique rétro-futuristeLa bande originale des aventures d'UlysseJ'habiterais dans les abysses j'aurais pas plus de pressionTout ce que je veux : foutre le feu dans ma ville #NéronDonner mon corps à la science inclut la dissectionDes jours entiers je récite mes leçons rourmenté dans une p

### Character : "

In [155]:
print(re.findall('.*".*', corpus))

corpus = re.sub('"', "", corpus)

['J\'essaie d\'leur rendre la guerre un peu moins horrible Comme dans "La vie est belle" d\'Roberto Benigni', 'J\'te souhaite la bienvenue au Havre , autrefois appelé "Franciscopolis"']


### \(

In [156]:
corpus = re.sub("\(", "", corpus)

## »|«

In [157]:
print(re.findall(".*»|«.*", corpus))

corpus = re.sub("»|«", "", corpus)

['« Pourquoi détruire les maisons des Palestiniens Après tout cette terre leur appartient!', '« Mes parents sont insensibles et cruels', "C'est décidé! Il faut que je rentre chez moi »", '« Médine Médine', 'Médine Médine »', '« Médine Médine', 'Médine Médine »', '« Médine Médine', 'Médine Médine »', '« Double discours', 'Petit journaliste en stage »', '« Tu te rapproches de la ratonnade bicot', "Voudrais-tu t'attirer les foudres? »"]


## \x80

In [158]:
print(re.findall(".*\x80.*", corpus))

corpus = re.sub("\x80", "", corpus)

["Nous n'voulions pas non plus d\x80'une Algérie Française", "Lorsque ma part algérienne s'\x80\x99exprime dans le micro d'\x80\x99la vie"]


## \x99

In [159]:
print(re.findall(".*\x99.*", corpus))

corpus = re.sub("\x99", "", corpus)

["Lorsque ma part algérienne s'\x99exprime dans le micro d'\x99la vie"]


### '

In [160]:
re.findall("'\s", corpus)[:10]

["' ", "'\n", "'\n", "'\n", "' ", "'\n", "' ", "' ", "' ", "'\n"]

In [161]:
def remove_special(match):
    return match.group(0).replace("'", "")

corpus = re.sub(r"'\s|\s+'", remove_special, corpus, flags=re.DOTALL)

In [162]:
re.findall(".*Ā.*", corpus)

['Ou finira comme les peuples des ʿĀd et des Thamūd']

## \ufeff

In [163]:
print(re.findall(".*\ufeff.*", corpus))

corpus = re.sub("\ufeff", "", corpus)

['Proofetmaître Henni\ufeff-Mansour']


### ʿ

In [164]:
print(re.findall(".*ʿ.*", corpus))

corpus = re.sub("ʿ", "", corpus)

['Ou finira comme les peuples des ʿĀd et des Thamūd']


### ·

In [165]:
print(re.findall(".*·.*", corpus))

corpus = re.sub("·", "", corpus)

['Propriété intellectuelle TA3 les convaincu·es', 'Entre nous, entre convaincu·es, pas de danse du ventre']


In [166]:
re.findall(r"\w+ğ\w+", corpus)

['Erdoğan']

In [167]:
print(len(set(corpus)),sorted(set(corpus)))

114 ['\n', ' ', '!', '%', '&', "'", ')', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '§', 'À', 'Ç', 'È', 'É', 'Ê', 'Ô', 'Ö', 'à', 'á', 'â', 'ã', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'í', 'î', 'ï', 'ñ', 'ô', 'ö', 'ù', 'û', 'ü', 'Ā', 'ā', 'ğ', 'ō', 'Œ', 'œ', 'ū', '–']


Ok the txt seems good for now

In [168]:
with open(f"clean_corpus_Medine.txt", "w", encoding="utf-8") as f :
    f.write(corpus)